In [10]:
import os
import json
from qdrant_client import QdrantClient, models
from dotenv import load_dotenv

In [12]:
load_dotenv()
API_KEY = os.getenv("API_KEY_QDRANT")
URL = "https://d98e588d-5738-4671-ba4a-32e50e71f68d.europe-west3-0.gcp.cloud.qdrant.io:6333"
COLLECTION_NAME = "recipe_rag_collection"

In [8]:
client = QdrantClient(
    url=URL,
    api_key=API_KEY,
)

In [11]:
from qdrant_client.models import Distance, VectorParams

VECTOR_DIMENSION = 3072

client.recreate_collection(
    collection_name=COLLECTION_NAME,
    vectors_config=VectorParams(size=VECTOR_DIMENSION, distance=Distance.COSINE)
)


/tmp/ipykernel_80664/2672412017.py:6: DeprecationWarning: `recreate_collection` method is deprecated and will be removed in the future. Use `collection_exists` to check collection existence and `create_collection` instead.
  client.recreate_collection(


True

In [12]:
with open('../data/dataset_embeddings.json', 'r', encoding='utf8') as f:
    data_points = json.load(f)

In [13]:
points_to_upsert = []

for idx, item in enumerate(data_points):
    point = {
        "id": idx,
        "vector": item['embedding'],
        "payload": {
            "title": item['title'],
            "ingredients": item['ingredients'],
            "NER": item['NER'],
            "directions": item['directions']
        }
    }
    points_to_upsert.append(point)

In [14]:
batch_size = 100

In [15]:
for i in range(0, len(points_to_upsert), batch_size):
    batch = points_to_upsert[i:i+batch_size]
    client.upsert(
        collection_name=COLLECTION_NAME,
        points=batch,
        wait=True
    )

In [13]:
client.create_payload_index(
    collection_name=COLLECTION_NAME,
    field_name="title",
    field_schema=models.TextIndexParams(
        type=models.TextIndexType.TEXT,
        tokenizer=models.TokenizerType.WORD,
        lowercase=True,
    )
)


UpdateResult(operation_id=101, status=<UpdateStatus.COMPLETED: 'completed'>)